# Loom — kernel for multi-agent LLM rooms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hdubey-debug/loom/blob/main/examples/colab_demo.ipynb)

Loom is a small Python kernel that turns ordinary callables (any LLM, any wrapper, any scripted function) into *agents* sharing a single message bus. The kernel handles event ordering, lease arbitration, obligation tracking, dead-letter routing, and a journal. **You** drop in a `ConversationPolicy` to decide *who* may speak, *when*, and *what extra instructions* to render.

This notebook walks through the four bundled policies plus one custom policy that uses a v0.2 hook. Each demo uses three deterministic mock agents — no API keys required.

**Cells run top to bottom.** Total runtime ~30s (mostly the install).

---

Repo: https://github.com/hdubey-debug/loom

## 0. Install

Loom is pure Python; no GPU, no provider keys. Install takes ~30 seconds in Colab.

In [ ]:
!pip install -q git+https://github.com/hdubey-debug/loom.git

## 1. Setup — three mock agents + a result printer

Real Loom agents are anything matching the `Agent` protocol — `id` plus a `stream(prompt) -> Iterator[str]`. The bundled `agent_from_send` adapter wraps a one-shot `prompt -> str` callable. Each mock agent below has a fixed personality so you can see, demo by demo, *which* agents the policy let speak.

In [ ]:
from loom import LoomRoom, agent_from_send

def alice_send(prompt: str) -> str:
    return "Alice: I'd start by clarifying what success looks like."

def bob_send(prompt: str) -> str:
    return "Bob: Counterpoint — test the binding constraint before planning."

def carol_send(prompt: str) -> str:
    return "Carol: Synthesis — name the strongest version of each side, then choose."

alice = agent_from_send("alice", alice_send, persona="planner")
bob   = agent_from_send("bob",   bob_send,   persona="critic")
carol = agent_from_send("carol", carol_send, persona="synthesiser")

def show(result, label=""):
    """Pretty-print a TurnResult."""
    print(f"=== {label} ===")
    print(f"  routing      : {result.routing_case}")
    print(f"  closed_reason: {result.closed_reason}")
    print(f"  replies      : {len(result.messages)}")
    for m in result.messages:
        print(f"    [{m.sender}] {m.body}")
    print()

## 2. `OpenChatPolicy` — broadcast to everyone

The simplest non-trivial policy: every active capable participant gets to respond. With three agents and one user post, expect three replies in arrival order.

Note the `with room:` context manager — actor threads start on `__enter__` and stop on `__exit__`. Posting to a room that isn't started returns nothing because no actor is listening.

In [ ]:
from loom import OpenChatPolicy

with LoomRoom(agents=[alice, bob, carol], policy=OpenChatPolicy()) as room:
    result = room.post_and_wait("Should we ship the new feature this week?")
    show(result, "OpenChatPolicy — all three should respond")

## 3. `SingleResponderPolicy` — route everything to one agent

Use this for a chatbot / single-model copilot shape: the policy always routes to one configured pid. The other two agents see the user post but the kernel rejects their lease attempts.

In [ ]:
from loom import SingleResponderPolicy

with LoomRoom(agents=[alice, bob, carol],
              policy=SingleResponderPolicy("bob")) as room:
    result = room.post_and_wait("Should we ship the new feature this week?")
    show(result, "SingleResponderPolicy(bob) — only Bob should respond")

## 4. `RoundRobinPolicy` — strict rotation

Each user post advances the rotation pointer by exactly one speaker. Three posts cycle through `[alice, bob, carol]`. The pointer survives across posts within the same `with room:` block.

In [ ]:
from loom import RoundRobinPolicy

with LoomRoom(agents=[alice, bob, carol],
              policy=RoundRobinPolicy(["alice", "bob", "carol"])) as room:
    for i in range(3):
        result = room.post_and_wait(f"Round {i+1}: anything to add?")
        show(result, f"RoundRobinPolicy — turn {i+1}")

## 5. `DefaultPolicy` — `@-mention` for direct routing

`DefaultPolicy` is the production-grade one: vocative detection, fallback chains, and direct-mention priority. An `@<id>` in the user's message routes that turn straight to that participant.

In [ ]:
from loom import DefaultPolicy

with LoomRoom(agents=[alice, bob, carol], policy=DefaultPolicy()) as room:
    result = room.post_and_wait("@alice what's your read on this?")
    show(result, "DefaultPolicy + @alice — direct route")

    result = room.post_and_wait("@carol you wrap us up.")
    show(result, "DefaultPolicy + @carol — direct route")

## 6. Custom policy — using a v0.2 hook

v0.2 added several optional hooks on `ConversationPolicy`. The simplest one to demo is `charter_text` — a string the kernel injects into every prompt's preamble immediately after `LOOM_PROTOCOL_INSTRUCTIONS` (which is fixed and unconditional).

Here we subclass `OpenChatPolicy` and add a brevity rule. With a real LLM you'd see all three agents follow the rule. Mock agents ignore prompts, so the *visible* effect is that the rule is rendered into the prompt — but the routing is still broadcast.

Other v0.2 hooks you can override the same way:

- `dead_letter_target(state, removed_participant)` — pick the reroute target when an `@`-mentioned agent is removed mid-turn.
- `should_post_response(body, state, participant_id)` — veto a draft after the kernel's filters (idle-phrase, IoU loop-guard) have passed.
- `prompt_sections(state, participant_id, trigger_event)` — inject named sections late in the system preamble.

Plus pluggable `RoomConfig.lease_checks` and `RoomConfig.trigger_priority` for advanced routing.

In [ ]:
from loom import OpenChatPolicy

class StrictBrevityPolicy(OpenChatPolicy):
    """OpenChat plus a one-line charter rule."""

    def charter_text(self, state):
        return ("BREVITY RULE: every reply must be a single sentence, "
                "max 15 words. No bullet lists.")

with LoomRoom(agents=[alice, bob, carol],
              policy=StrictBrevityPolicy()) as room:
    result = room.post_and_wait("How do we ship faster?")
    show(result, "Custom StrictBrevityPolicy — charter rule injected")

## 7. Bonus — peek at the bus

Loom journals every event into an append-only log. Below we reach into the underlying session (the `_session` attr is intentionally semi-private — this is for inspection only) and dump every event posted during one turn.

You'll see the user post, control events (`user_turn_opened`, `obligation_recorded`, lease grants), stream events (`start` / `delta` / `end`), the committed `chat` events, and the closing control event. This is also exactly what the journal would write to disk for replay.

In [ ]:
with LoomRoom(agents=[alice, bob, carol], policy=OpenChatPolicy()) as room:
    result = room.post_and_wait("ping")
    bus = room._session.bus
    print(f"bus length: {len(bus)} events\n")
    for ev in bus.snapshot():
        body = str(ev.body)
        if len(body) > 70:
            body = body[:67] + "..."
        print(f"  id={ev.id:3d}  kind={ev.kind:7s}  sender={ev.sender:7s}  {body}")

## 8. Where to next

- **Plug in a real LLM**: replace any `*_send` callable with one that calls Anthropic / OpenAI / your provider. The policy code stays identical. See `examples/openai_two_agents.py` in the repo for a template.
- **DM channels**: post on `channel="dm:alice"` to send a private sidebar visible only to alice + user + system. Useful for clarifications without polluting the main thread.
- **Custom `LeaseCheck`**: write your own gate ("agents must hold an active certification slot") and pass it in `RoomConfig(lease_checks=(MyCheck(), *DEFAULT_LEASE_CHECKS))`.
- **Roadmap**: v0.3 adds a *controller* mechanism (one agent's posts open chained turns — a CEO/orchestrator pattern); v0.4 adds structured `tool_call` / `tool_result` events; v0.5 explores Claude Code workers.

Star the repo if this was useful: https://github.com/hdubey-debug/loom